In [2]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
import numpy as np
import os
import pandas as pd 

In [3]:
import pandas as pd
use_cols = ['Created Date','Closed Date','Complaint Type','Agency','Borough','Descriptor']
df = pd.read_csv('../dsml-project-team/311-service-requests-from-2010-to-present.csv', usecols=use_cols, nrows=500000)


In [4]:
df

,Created Date,Closed Date,Agency,Complaint Type,Descriptor,Borough
0,2019-12-01T02:04:01.000,NaN,DOT,Street Condition,Pothole,MANHATTAN
1,2019-12-01T01:59:41.000,NaN,NYPD,Noise - Commercial,Loud Music/Party,BROOKLYN
2,2019-12-01T01:59:08.000,NaN,NYPD,Noise - Residential,Loud Music/Party,BROOKLYN
3,2019-12-01T01:58:23.000,NaN,NYPD,Noise - Residential,Loud Music/Party,QUEENS
4,2019-12-01T01:58:07.000,NaN,NYPD,Illegal Parking,Commercial Overnight Parking,QUEENS
...,...,...,...,...,...,...
499995,2019-09-22T11:39:38.000,2019-09-22T20:09:28.000,NYPD,Blocked Driveway,Partial Access,QUEENS
499996,2019-09-22T11:39:25.000,2019-09-26T13:02:49.000,DPR,Overgrown Tree/Branches,Hitting Building,BRONX
499997,2019-09-22T11:39:20.000,NaN,TLC,For Hire Vehicle Complaint,Driver Complaint - Non Passenger,MANHATTAN
499998,2019-09-22T11:38:28.000,2019-09-22T12:16:31.000,NYPD,Noise - Residential,Banging/Pounding,BROOKLYN


In [5]:
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'], errors='coerce')
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords'); nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['Complaint Type'].astype(str).apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [6]:
df['clean_text']

0               street condition
1               noise commercial
2              noise residential
3              noise residential
4                illegal parking
                   ...          
499995          blocked driveway
499996    overgrown treebranches
499997    hire vehicle complaint
499998         noise residential
499999              damaged tree
Name: clean_text, Length: 500000, dtype: object

TF-TDI Vectorization

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [8]:
# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean_text'])
y = df['Agency']

In [9]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Train
clf = LogisticRegression(max_iter=200, class_weight='balanced')
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,200
,multi_class,'deprecated'


In [10]:
# Evaluate
y_pred = clf.predict(X_test)

In [11]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         DCA       1.00      1.00      1.00       548
         DEP       1.00      0.99      1.00      7009
         DHS       1.00      1.00      1.00      1400
         DOB       0.99      0.98      0.98      4277
         DOE       1.00      1.00      1.00        88
       DOHMH       0.99      1.00      0.99      2682
       DOITT       1.00      1.00      1.00        17
         DOT       1.00      1.00      1.00     10436
         DPR       1.00      1.00      1.00      3893
        DSNY       1.00      1.00      1.00      4553
         EDC       1.00      1.00      1.00       264
         HPD       0.99      1.00      1.00     16669
        NYPD       1.00      1.00      1.00     44715
         TLC       1.00      1.00      1.00      3449

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000



In [12]:
print(confusion_matrix(y_test, y_pred))

[[  548     0     0     0     0     0     0     0     0     0     0     0
      0     0]
 [    0  6971     0     0     0    38     0     0     0     0     0     0
      0     0]
 [    0     0  1400     0     0     0     0     0     0     0     0     0
      0     0]
 [    0     0     0  4174     0     0     0     0     0     0     0   103
      0     0]
 [    0     0     0     0    88     0     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0  2682     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0    17     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0 10436     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0     0  3893     0     0     0
      0     0]
 [    0     0     0     0     0     2     0     0     0  4551     0     0
      0     0]
 [    0     0     0     0     0     0     0     0     0     0   264     0
      0     0]
 [    0     0     0  

Save model & Vectorizer

In [13]:
import joblib

joblib.dump(clf, "department_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']